In [ ]:
# hmrc data pipeline scrape test

# This script is used to test the scraping of HMRC data for the data pipeline
#  following successful test of single text file


In [1]:
# Standard libraries for web scraping, file handling, and timing
import requests      # For downloading web pages and files
import zipfile       # For extracting zip archives
import io            # For handling in-memory byte streams
import csv           # For writing CSV output
import time          # For timing the process
import re            # For regular expressions (pattern matching in HTML)
from pathlib import Path  # For clean file path handling

# Set up folders for our work
# Adjust these paths to match your project structure
base_folder = Path(r"C:\Users\tcolw\Documents\local_coding_projs\for_fun_weekly_projects\12_py_pipeline")
data_folder = base_folder / "data_2026"
data_folder.mkdir(exist_ok=True)  # Create the folder if it doesn't exist

# URL
archive_url = "https://www.uktradeinfo.com/trade-data/latest-bulk-data-sets/bulk-data-sets-archive"

In [6]:
# Cell 2 – Fetch the archive page and extract 2026 Exports zip link

print("Fetching archive page...")
response = requests.get(archive_url)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Find the Exports div by exact ID
exports_div = soup.find("div", id="exports-(bds-exp-yymm)")

# Safety check: make sure the div exists
if not exports_div:
    raise RuntimeError(
        "Could not find div with id='exports-(bds-exp-yymm)'. "
        "The webpage structure may have changed."
    )

print(f"Found Exports container: {exports_div.get('id')}")

# Get all zip links inside this div
zip_links = [
    (a["href"], a.get_text(strip=True))
    for a in exports_div.find_all("a", href=True)
    if a["href"].lower().endswith(".zip")
]

print(f"\nFound {len(zip_links)} zip files in Exports section:")
for url, text in zip_links:
    print(f"  URL: {url}")
    print(f"  Text: {text}")
    print()

# Filter for 2026
# The URL might be like "bdsexp_26archive.zip" (26 = 2026)
# So we check for either "2026" or "_26" in the URL
exports_2026 = [(url, text) for url, text in zip_links if "2026" in url or "_26" in url]

print(f"Filtered to {len(exports_2026)} link(s) for 2026:")
for url, text in exports_2026:
    print(f"  - {text}: {url}")

if not exports_2026:
    # If still nothing found, just take the first zip (might be the only one)
    print("\nWARNING: No 2026-specific zip found. Using first available zip.")
    if zip_links:
        exports_2026 = [zip_links[0]]
    else:
        raise RuntimeError("No zip files found in the Exports section at all")

zip_url = exports_2026[0][0]

# Make absolute URL if needed
if zip_url.startswith("/"):
    zip_url = "https://www.uktradeinfo.com" + zip_url
elif zip_url.startswith("http"):
    pass  # Already absolute
else:
    # Relative path without leading slash
    zip_url = "https://www.uktradeinfo.com" + "/" + zip_url

print(f"\nUsing: {zip_url}")

Fetching archive page...
Found Exports container: exports-(bds-exp-yymm)

Found 11 zip files in Exports section:
  URL: /media/xm1jtft5/bdsexp_26archive.zip
  Text: Exports: 2026
                            (ZIP, 13.6 MB)

  URL: /media/ye2mtvjj/bdsexp_25archive.zip
  Text: Exports: 2025
                            (ZIP, 33.4 MB)

  URL: /media/gjmhgw2g/bdsexp_24archive.zip
  Text: Exports: 2024
                            (ZIP, 34 MB)

  URL: /media/br3dcbef/bdsexp_23archive.zip
  Text: Exports: 2023
                            (ZIP, 34.3 MB)

  URL: /media/0e1b11zq/bdsexp_22archive.zip
  Text: Exports: 2022
                            (ZIP, 35.4 MB)

  URL: /media/ifsljkie/bdsexp_21archive.zip
  Text: Exports: 2021
                            (ZIP, 35.1 MB)

  URL: /media/gw2oqtjj/bdsexp_20archive.zip
  Text: Exports: 2020
                            (ZIP, 28.2 MB)

  URL: /media/p0fjm54d/bdsexp_19archive.zip
  Text: Exports: 2019
                            (ZIP, 30.5 MB)

  URL: /m

In [7]:
zip_links

[('/media/xm1jtft5/bdsexp_26archive.zip',
  'Exports: 2026\r\n                            (ZIP, 13.6 MB)'),
 ('/media/ye2mtvjj/bdsexp_25archive.zip',
  'Exports: 2025\r\n                            (ZIP, 33.4 MB)'),
 ('/media/gjmhgw2g/bdsexp_24archive.zip',
  'Exports: 2024\r\n                            (ZIP, 34 MB)'),
 ('/media/br3dcbef/bdsexp_23archive.zip',
  'Exports: 2023\r\n                            (ZIP, 34.3 MB)'),
 ('/media/0e1b11zq/bdsexp_22archive.zip',
  'Exports: 2022\r\n                            (ZIP, 35.4 MB)'),
 ('/media/ifsljkie/bdsexp_21archive.zip',
  'Exports: 2021\r\n                            (ZIP, 35.1 MB)'),
 ('/media/gw2oqtjj/bdsexp_20archive.zip',
  'Exports: 2020\r\n                            (ZIP, 28.2 MB)'),
 ('/media/p0fjm54d/bdsexp_19archive.zip',
  'Exports: 2019\r\n                            (ZIP, 30.5 MB)'),
 ('/media/lvwa0fbz/bdsexp_18archive.zip',
  'Exports: 2018\r\n                            (ZIP, 30.2 MB)'),
 ('/media/y11fd51z/bdsexp_17ar

In [8]:
# Download the zip file
zip_filename = data_folder / "Exports_2026.zip"

print(f"Downloading 2026 Exports zip from: {zip_url}")
start_download = time.time()

zip_response = requests.get(zip_url)
zip_response.raise_for_status()

# Save the zip file to disk
with open(zip_filename, "wb") as f:
    f.write(zip_response.content)

download_time = time.time() - start_download
print(f"Downloaded to: {zip_filename}")
print(f"Download time: {download_time:.2f} seconds")
print(f"File size: {zip_filename.stat().st_size / (1024*1024):.2f} MB")

Downloaded to: C:\Users\tcolw\Documents\local_coding_projs\for_fun_weekly_projects\12_py_pipeline\data_2026\Exports_2026.zip
Download time: 1.88 seconds
File size: 13.62 MB


In [10]:
# Open the zip and show what's inside
print("Contents of the zip file:")
with zipfile.ZipFile(zip_filename, "r") as zf:
    file_list = zf.namelist()
    for name in file_list:
        print(f"  - {name}")
    
    

Contents of the zip file:
  - BDSexp2601.txt
  - BDSexp2602.txt
  - BDSexp2603.txt
  - BDSexp2604.txt
  - BDSexp2605.txt


In [11]:
# Field layout from the UK Trade Info technical specifications
# Each tuple: (column_name, start_position, end_position)
# Positions are 1-based and inclusive, as in the spec table
FIELDS = [
    ("PERREF",            1,   6),  # Period Reference (YYYYMM)
    ("TYPE",              7,   7),  # Record type
    ("MONTHAC",           8,  13),  # Month of Account
    ("COMCODE",          14,  21),  # Commodity code (8 digits)
    ("SITC",             22,  26),  # SITC code (5 digits)
    ("COD_SEQ",          27,  29),  # Country code (numeric)
    ("COD_ALPHA",        30,  31),  # Country code (alpha)
    ("PORT_SEQ",         32,  34),  # Port code (numeric)
    ("PORT_ALPHA",       35,  37),  # Port code (alpha)
    ("COO_SEQ",          38,  40),  # Country of Origin (numeric)
    ("COO_ALPHA",        41,  42),  # Country of Origin (alpha)
    ("MODE_OF_TRANSPORT",43,  44),  # Mode of transport
    ("STAT_VALUE",       45,  56),  # Statistical value
    ("NET_MASS",         57,  68),  # Net mass (kg)
    ("SUPP_UNIT",        69,  80),  # Supplementary unit
    ("SUPPRESSION",      81,  81),  # Suppression indicator
    ("FLOW",             82,  84),  # Import/Export flow
    ("REC_TYPE",         85,  85),  # Record type flag
]

# Build the list of column names for the CSV header
fieldnames = [name for name, _, _ in FIELDS]
print(f"Field names: {fieldnames}")

Field names: ['PERREF', 'TYPE', 'MONTHAC', 'COMCODE', 'SITC', 'COD_SEQ', 'COD_ALPHA', 'PORT_SEQ', 'PORT_ALPHA', 'COO_SEQ', 'COO_ALPHA', 'MODE_OF_TRANSPORT', 'STAT_VALUE', 'NET_MASS', 'SUPP_UNIT', 'SUPPRESSION', 'FLOW', 'REC_TYPE']


In [12]:
def parse_line(line):
    """
    Parse a single fixed-width line into a dictionary of field values.
    """
    row = {}
    
    for name, start, end in FIELDS:
        # Convert 1-based inclusive positions to 0-based Python slices
        start0 = start - 1
        end0 = end  # Python slice end is exclusive
        
        # Extract the substring (safely handle short lines)
        if len(line) >= end0:
            value = line[start0:end0]
        else:
            value = line[start0:]
        
        row[name] = value
    
    return row



In [13]:
# Quick test (optional)
test_line = "202602120260212345678"[:85]  # Dummy line
print(f"Test parse: {parse_line(test_line)}")

Test parse: {'PERREF': '202602', 'TYPE': '1', 'MONTHAC': '202602', 'COMCODE': '12345678', 'SITC': '', 'COD_SEQ': '', 'COD_ALPHA': '', 'PORT_SEQ': '', 'PORT_ALPHA': '', 'COO_SEQ': '', 'COO_ALPHA': '', 'MODE_OF_TRANSPORT': '', 'STAT_VALUE': '', 'NET_MASS': '', 'SUPP_UNIT': '', 'SUPPRESSION': '', 'FLOW': '', 'REC_TYPE': ''}


In [17]:
# Main processing step: read all monthly files from the zip, parse, and combine

output_csv = data_folder / "BDSExp2026_full.csv"

print("Starting full year processing...")
start_processing = time.time()

# Track statistics
total_rows = 0
monthly_row_counts = {}

# Open the zip file
with zipfile.ZipFile(zip_filename, "r") as zf:
    # Get all text files (monthly data files)
    txt_files = sorted([name for name in zf.namelist() if name.endswith(".txt")])
    
    print(f"Processing {len(txt_files)} monthly files...")
    
    # Open the output CSV file for writing
    with open(output_csv, "w", encoding="utf-8", newline="") as f_out:
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()
        
        # Process each monthly file
        for txt_file in txt_files:
            print(f"  Processing: {txt_file}")
            month_start = time.time()
            month_rows = 0
            
            # Read the text file from the zip
            with zf.open(txt_file) as f_in:
                for line_bytes in f_in:
                    # Decode bytes to string
                    line = line_bytes.decode("utf-8", errors="ignore").rstrip("\n\r")
                    
                    # Skip empty lines
                    if not line.strip():
                        continue
                    
                    # Parse the fixed-width line
                    row = parse_line(line)
                    
                    # Write to CSV
                    writer.writerow(row)
                    
                    month_rows += 1
                    total_rows += 1
            
            month_time = time.time() - month_start
            monthly_row_counts[txt_file] = month_rows
            print(f"    -> {month_rows:,} rows in {month_time:.2f} seconds")

processing_time = time.time() - start_processing

print(f"\n{'='*60}")
print(f"COMPLETE!")
print(f"{'='*60}")
print(f"Output file: {output_csv}")
print(f"Total rows: {total_rows:,}")
print(f"Total processing time: {processing_time:.2f} seconds")
print(f"\nRows per month:")
for filename, count in sorted(monthly_row_counts.items()):
    print(f"  {filename}: {count:,} rows")

Starting full year processing...
Processing 5 monthly files...
  Processing: BDSexp2601.txt
    -> 241,605 rows in 3.06 seconds
  Processing: BDSexp2602.txt
    -> 257,031 rows in 3.16 seconds
  Processing: BDSexp2603.txt
    -> 261,438 rows in 2.69 seconds
  Processing: BDSexp2604.txt
    -> 257,190 rows in 2.35 seconds
  Processing: BDSexp2605.txt
    -> 260,169 rows in 2.32 seconds

COMPLETE!
Output file: C:\Users\tcolw\Documents\local_coding_projs\for_fun_weekly_projects\12_py_pipeline\data_2026\BDSExp2026_full.csv
Total rows: 1,277,433
Total processing time: 13.59 seconds

Rows per month:
  BDSexp2601.txt: 241,605 rows
  BDSexp2602.txt: 257,031 rows
  BDSexp2603.txt: 261,438 rows
  BDSexp2604.txt: 257,190 rows
  BDSexp2605.txt: 260,169 rows


In [27]:
# Optional: Load and preview the sample data
import pandas as pd

df_sample = pd.read_csv(output_csv)

print(f"Sample data shape: {df_sample.shape}")
print(f"\nColumn types:")
print(df_sample.dtypes)
print(f"\nFirst 10 rows:")
df_sample.head(10)

Sample data shape: (1277433, 18)

Column types:
PERREF                int64
TYPE                  int64
MONTHAC               int64
COMCODE              object
SITC                 object
COD_SEQ               int64
COD_ALPHA               str
PORT_SEQ              int64
PORT_ALPHA              str
COO_SEQ                 str
COO_ALPHA               str
MODE_OF_TRANSPORT    object
STAT_VALUE            int64
NET_MASS             object
SUPP_UNIT            object
SUPPRESSION           int64
FLOW                    str
REC_TYPE              int64
dtype: object

First 10 rows:


C:\Users\tcolw\AppData\Local\Temp\ipykernel_31168\1194511668.py:4: DtypeWarning: Columns (0: COMCODE, 1: SITC, 2: MODE_OF_TRANSPORT, 3: NET_MASS, 4: SUPP_UNIT) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sample = pd.read_csv(output_csv)


,PERREF,TYPE,MONTHAC,COMCODE,SITC,COD_SEQ,COD_ALPHA,PORT_SEQ,PORT_ALPHA,COO_SEQ,COO_ALPHA,MODE_OF_TRANSPORT,STAT_VALUE,NET_MASS,SUPP_UNIT,SUPPRESSION,FLOW,REC_TYPE
0,202601,1,202608,01012100,00150,1,FR,7,DOV,,,60,10000,500,1,0,exp,0
1,202601,1,202608,01012100,00150,1,FR,28,PTM,,,60,105686,5500,3,0,exp,0
2,202601,1,202608,01012100,00150,1,FR,381,DEU,,,60,2487103,2500,5,0,exp,0
3,202601,1,202608,01012100,00150,1,FR,382,EUT,,,60,5500,1000,2,0,exp,0
4,202601,1,202608,01012100,00150,3,NL,7,DOV,,,60,88277,2500,5,0,exp,0
5,202601,1,202608,01012100,00150,4,DE,7,DOV,,,60,15000,1000,2,0,exp,0
6,202601,1,202608,01012100,00150,4,DE,381,DEU,,,60,20475,1033,4,0,exp,0
7,202601,1,202608,01012100,00150,7,IE,7,DOV,,,60,1000,500,1,0,exp,0
8,202601,1,202608,01012100,00150,7,IE,28,PTM,,,60,2500,500,1,0,exp,0
9,202601,1,202608,01012100,00150,7,IE,101,MIL,,,60,1000,500,1,0,exp,0


In [ ]:
# Final summary
print(f"\n{'='*60}")
print(f"FINAL SUMMARY")
print(f"{'='*60}")
print(f"Year: 2026")
print(f"Full dataset: {output_csv}")
print(f"  - Total rows: {total_rows:,}")
print(f"  - File size: {output_csv.stat().st_size / (1024*1024):.2f} MB")



FINAL SUMMARY
Year: 2026
Full dataset: C:\Users\tcolw\Documents\local_coding_projs\for_fun_weekly_projects\12_py_pipeline\data_2026\BDSExp2026_full.csv
  - Total rows: 1,277,433
  - File size: 126.70 MB

Sample dataset: C:\Users\tcolw\Documents\local_coding_projs\for_fun_weekly_projects\12_py_pipeline\data_2026\BDSExp2026_sample_10k.csv
  - Rows: 0

Monthly breakdown:
